# Himawari-8 CLP Datetime Processing

Process both consolidated datasets in memory. The original CSV files are not modified.

In [1]:
from pathlib import Path
from datetime import time

import pandas as pd

DATA_DIR = Path("dataset/row/himawari8/CLP")
PROJECT_ROOT = next(
    (
        root
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / DATA_DIR / "CLP_2016_2020.csv").is_file()
        and (root / DATA_DIR / "CLP_2026.csv").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate both Himawari-8 CSV files from {Path.cwd()}"
    )

CLP_2016_2020 = pd.read_csv(
    PROJECT_ROOT / DATA_DIR / "CLP_2016_2020.csv",
)
CLP_2026 = pd.read_csv(
    PROJECT_ROOT / DATA_DIR / "CLP_2026.csv",
)
dataset = [CLP_2016_2020, CLP_2026]

print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded {len(dataset)} datasets")

Project root: c:\FYP\coding
Loaded 2 datasets


## Task1: Datetime Processing

The Himawari-8 generation code defines `timestamp` as UTC and derives `malaysia_datetime` by adding eight hours. Verify that the existing Malaysia-local value agrees with converting UTC to `Asia/Kuala_Lumpur`; do not add another eight hours. Remove only `timestamp`; retain all other source columns.

In [2]:
for dataset_number, data in enumerate(dataset, start=1):
    source_timestamp = pd.to_datetime(
        data["timestamp"], errors="raise"
    ).dt.tz_localize("UTC")
    recorded_malaysia_datetime = pd.to_datetime(
        data["malaysia_datetime"], errors="raise"
    )
    if recorded_malaysia_datetime.dt.tz is not None:
        raise ValueError("Expected naive Malaysia-local datetime values in the CSV.")

    malaysia_datetime = recorded_malaysia_datetime.dt.tz_localize(
        "Asia/Kuala_Lumpur"
    )
    expected_malaysia_datetime = source_timestamp.dt.tz_convert(
        "Asia/Kuala_Lumpur"
    )
    if not malaysia_datetime.equals(expected_malaysia_datetime):
        raise ValueError(
            f"Source and Malaysia timestamps do not match in dataset {dataset_number}."
        )

    data["malaysia_datetime"] = malaysia_datetime
    data = (
        data.drop(columns="timestamp")
        .sort_values("malaysia_datetime", kind="stable")
        .reset_index(drop=True)
    )
    dataset[dataset_number - 1] = data

    if not data["malaysia_datetime"].is_monotonic_increasing:
        raise ValueError(f"Timestamps are not chronological in dataset {dataset_number}.")
    print(
        f"Dataset {dataset_number}: {len(data):,} rows; "
        f"columns={data.columns.tolist()}; "
        f"timezone={data['malaysia_datetime'].dt.tz}; "
        f"range={data['malaysia_datetime'].min()} to {data['malaysia_datetime'].max()}"
    )

CLP_2016_2020, CLP_2026 = dataset

Dataset 1: 107,552 rows; columns=['malaysia_datetime', 'latitude', 'longitude', 'Hour', 'CLOT', 'CLTT', 'CLTH', 'CLER_23', 'CLTYPE', 'QA']; timezone=Asia/Kuala_Lumpur; range=2016-01-01 09:00:00+08:00 to 2020-12-31 18:50:00+08:00
Dataset 2: 3,594 rows; columns=['malaysia_datetime', 'latitude', 'longitude', 'Hour', 'CLOT', 'CLTT', 'CLTH', 'CLER_23', 'CLTYPE', 'QA']; timezone=Asia/Kuala_Lumpur; range=2026-06-01 09:00:00+08:00 to 2026-07-31 18:50:00+08:00


## Task 2: Filter CLP_2026 to 10:00–17:00

Keep only observations whose Malaysia-local time is between 10:00:00 and 17:00:00, inclusive. `CLP_2016_2020` remains unchanged.

In [3]:
START_TIME = time(10, 0)
END_TIME = time(17, 0)

clp_2026_times = dataset[1]["malaysia_datetime"].dt.time
clp_2026_time_filter = clp_2026_times.between(
    START_TIME, END_TIME, inclusive="both"
)
CLP_2026 = dataset[1].loc[clp_2026_time_filter].reset_index(drop=True)
dataset[1] = CLP_2026

print(f"CLP_2026 rows before filtering: {len(clp_2026_times):,}")
print(f"CLP_2026 rows from 10:00 to 17:00 Malaysia time: {len(CLP_2026):,}")
print(
    "Included boundary rows: "
    f"10:00={int(CLP_2026['malaysia_datetime'].dt.time.eq(START_TIME).sum()):,}, "
    f"17:00={int(CLP_2026['malaysia_datetime'].dt.time.eq(END_TIME).sum()):,}"
)

CLP_2026 rows before filtering: 3,594
CLP_2026 rows from 10:00 to 17:00 Malaysia time: 2,558
Included boundary rows: 10:00=61, 17:00=61


## Task 3: Change the Time Interval

Aggregate each calendar date separately into right-closed intervals labeled by the interval end. For example, `(10:00, 11:00]` is labeled `11:00`. Continuous numeric measurements use arithmetic means, while `CLTYPE` and `QA` retain the last observed value. `CLP_2016_2020` uses 1-hour intervals; `CLP_2026` uses 10-minute intervals. Empty intervals are omitted, with no filling or interpolation.

In [4]:
RESAMPLE_FREQUENCIES = ["1h", "10min"]
CATEGORICAL_COLUMNS = ["CLTYPE", "QA"]
resampled_datasets = []

for data, frequency in zip(dataset, RESAMPLE_FREQUENCIES, strict=True):
    daily_results = []
    value_columns = [column for column in data.columns if column != "malaysia_datetime"]
    mean_columns = [
        column
        for column in value_columns
        if pd.api.types.is_numeric_dtype(data[column])
        and column not in CATEGORICAL_COLUMNS
    ]
    categorical_columns = [
        column for column in CATEGORICAL_COLUMNS if column in data.columns
    ]
    for _, day_data in data.groupby(data["malaysia_datetime"].dt.date, sort=True):
        resampler = day_data.set_index(
            "malaysia_datetime", drop=False
        ).resample(
            frequency,
            closed="right",
            label="right",
            origin="start_day",
        )
        interval_data = resampler[mean_columns].mean()
        interval_data["malaysia_datetime"] = interval_data.index
        for column in categorical_columns:
            interval_data[column] = resampler[column].last()
        interval_data = interval_data.loc[resampler.size().gt(0)]
        daily_results.append(interval_data.reset_index(drop=True))

    resampled_data = pd.concat(daily_results, ignore_index=True)
    resampled_data = resampled_data.loc[:, data.columns]
    if not resampled_data["malaysia_datetime"].is_monotonic_increasing:
        raise ValueError("Resampled timestamps are not in chronological order.")
    resampled_datasets.append(resampled_data)

dataset = resampled_datasets
CLP_2016_2020, CLP_2026 = dataset

print(f"CLP_2016_2020 (1-hour intervals): {len(CLP_2016_2020):,} rows")
print(f"CLP_2026 (10-minute intervals): {len(CLP_2026):,} rows")

CLP_2016_2020 (1-hour intervals): 20,081 rows
CLP_2026 (10-minute intervals): 2,558 rows


In [5]:
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for filename, data in zip(
    ["CLP_2016_2020_processed.csv", "CLP_2026_processed.csv"], dataset, strict=True
):
    output_file = PROCESSED_DIR / filename
    data.to_csv(output_file, index=False)
    print(f"Saved {len(data):,} rows to {output_file}")

Saved 20,081 rows to c:\FYP\coding\dataset\processed\CLP_2016_2020_processed.csv
Saved 2,558 rows to c:\FYP\coding\dataset\processed\CLP_2026_processed.csv
